# 3 - Auditing an ERP extract, and auditing the auditor

The financial dataset is real but anonymous. This notebook works with the other half of
the project: a synthetic SAP-style extract - vendors, materials, bills of materials and
purchase orders - generated with Faker and then deliberately corrupted.

Because every corruption is recorded in a ground-truth ledger, the quality engine can
be measured the way a detector should be, rather than merely described.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 100, "font.size": 9,
                     "axes.spines.top": False, "axes.spines.right": False})

In [2]:
from dq_anomaly.data.erp_generator import generate_erp_dataset
from dq_anomaly.data.defect_injector import DEFAULT_DEFECT_SPECS, inject_defects

clean = generate_erp_dataset(seed=42)
dirty, ledger = inject_defects(clean, seed=1337)

for name, frame in clean.items():
    print(f"{name:18s} {len(frame):>7,} rows x {frame.shape[1]:>2} columns")
print(f"\n{len(ledger):,} defects injected across {ledger['defect_code'].nunique()} families")

vendors                300 rows x  8 columns
materials            5,000 rows x 10 columns
bom_headers          1,200 rows x  7 columns
bom_lines            8,923 rows x  7 columns
purchase_orders     20,000 rows x  9 columns
po_lines            89,649 rows x 12 columns

4,665 defects injected across 14 families


In [3]:
clean["po_lines"].head(3)

,po_line_id,po_id,position,material_id,quantity,uom,unit_price,net_value,currency,delivery_date,goods_receipt_qty,invoice_qty
0,POL-0000001,PO-4500000001,10,MAT-003293,30.02,EA,64.40,1933.29,EUR,2024-11-01,30.02,26.17
1,POL-0000002,PO-4500000001,20,MAT-002980,6.79,M,18.84,127.92,EUR,2024-10-29,6.79,6.73
2,POL-0000003,PO-4500000001,30,MAT-002265,19.04,EA,8.90,169.46,EUR,2024-10-28,19.04,18.47


## What was broken, and why those things

In [4]:
specs = pd.DataFrame([{
    "code": s.code, "table": s.table, "column": s.column,
    "dimension": s.dimension, "rate": f"{s.rate:.1%}", "description": s.description,
} for s in DEFAULT_DEFECT_SPECS])
specs

,code,table,column,dimension,rate,description
0,MISSING_CRITICAL,po_lines,unit_price,completeness,1.2%,Unit price blanked out on purchase order lines
1,MISSING_OPTIONAL,vendors,tax_id,completeness,3.0%,Vendor tax id missing
2,TYPE_MISMATCH,po_lines,quantity,validity,0.5%,Quantity stored as European-formatted text
3,NEGATIVE_QUANTITY,po_lines,quantity,validity,0.4%,Negative ordered quantity
4,ZERO_PRICE,po_lines,unit_price,validity,0.6%,Unit price of zero
5,OUT_OF_RANGE_PRICE,materials,standard_price,plausibility,0.3%,"Fat-fingered standard price, orders of magnitu..."
6,INVALID_CATEGORY,vendors,payment_terms,validity,2.0%,Payment terms outside the controlled vocabulary
7,INVALID_UOM,po_lines,uom,consistency,0.8%,Line unit of measure disagrees with the materi...
8,DUPLICATE_ROW,purchase_orders,NaN,uniqueness,0.5%,Purchase order header duplicated with the same...
9,BROKEN_FK,po_lines,material_id,consistency,0.4%,Line points at a material that does not exist


These are not random mutations. Each one is a failure mode that actually occurs in
an ERP extract: a locale-confused export writing `1.234,00` into a numeric column, a
goods receipt posted against a deleted material, a payment-terms field filled by hand
so `NET30` arrives as `net30` and `NET 30 days`, a purchase order duplicated by a
re-run of the load job.

In [5]:
ledger.groupby(["table", "defect_code"]).size().rename("injected").reset_index()

,table,defect_code,injected
0,materials,ENCODING_NOISE,50
1,materials,OUT_OF_RANGE_PRICE,15
2,po_lines,ARITHMETIC_INCONSISTENCY,807
3,po_lines,BROKEN_FK,359
4,po_lines,INVALID_UOM,717
5,po_lines,MISSING_CRITICAL,1076
6,po_lines,NEGATIVE_QUANTITY,359
7,po_lines,TYPE_MISMATCH,448
8,po_lines,ZERO_PRICE,538
9,purchase_orders,DATE_ORDER_VIOLATION,141


## Scoring the corrupted batch

In [6]:
from dq_anomaly.quality.profile import load_profile
from dq_anomaly.quality.scorer import score_dataframe
from dq_anomaly.quality.defect_eval import (
    evaluate_defect_recall, evaluate_rule_precision, flagged_cells)

tables = ["po_lines", "purchase_orders", "vendors", "materials"]
findings, rows = [], []
for table in tables:
    profile = load_profile(f"profile_erp_{table}.yaml")
    report = score_dataframe(dirty[table], profile, reference_tables=dirty)
    findings.append(flagged_cells(dirty[table], report.rules, table))
    rows.append({
        "table": table,
        "geometric_score": round(report.global_score, 2),
        "arithmetic_score": round(report.global_score_arithmetic, 2),
        "grade": report.grade,
        "rules_failing": len(report.failed_rules),
    })
findings = pd.concat(findings, ignore_index=True)
pd.DataFrame(rows)

,table,geometric_score,arithmetic_score,grade,rules_failing
0,po_lines,99.77,99.77,A,11
1,purchase_orders,99.74,99.74,F,6
2,vendors,99.75,99.75,A,2
3,materials,99.95,99.95,A,2


Two things in that table are worth stopping on.

**`purchase_orders` scores 99.7 and still grades F.** A duplicated primary key is not a
gradeable condition: any join or aggregate on that table is now wrong, so the grade is
overridden regardless of how good the weighted score looks.

**The scores are high because the defect rates are low.** 4,665 genuine defects across
~115,000 rows is under 1%, so a headline score will always look reassuring. That is
precisely why the score is not the deliverable - the prioritized issue register is.

## Does the geometric mean actually behave differently?

In [7]:
from dq_anomaly.quality.scorer import aggregate_arithmetic, aggregate_geometric

weights = {"completeness": 0.25, "validity": 0.25, "consistency": 0.20,
           "uniqueness": 0.15, "plausibility": 0.15}
scenarios = {
    "evenly good": {"completeness": .95, "validity": .95, "consistency": .95,
                    "uniqueness": .95, "plausibility": .95},
    "one dimension dead": {"completeness": 1.0, "validity": 1.0, "consistency": 1.0,
                           "uniqueness": 0.0, "plausibility": 1.0},
    "keys half broken": {"completeness": 1.0, "validity": 1.0, "consistency": 1.0,
                         "uniqueness": 0.5, "plausibility": 1.0},
}
pd.DataFrame([{
    "scenario": name,
    "arithmetic": round(aggregate_arithmetic(s, weights), 1),
    "geometric": round(aggregate_geometric(s, weights), 1),
} for name, s in scenarios.items()])

,scenario,arithmetic,geometric
0,evenly good,95.0,95.0
1,one dimension dead,85.0,50.1
2,keys half broken,92.5,90.1


An arithmetic mean calls a batch with 100% duplicated keys an 85 - a solid B. The
geometric mean refuses to average that away. Quality dimensions are not substitutable:
being complete does not compensate for being unjoinable.

## Measuring the engine against the ground truth

In [8]:
recall = evaluate_defect_recall(ledger, findings)
recall

,table,defect_code,injected,detected,recall
0,materials,ENCODING_NOISE,50,50,1.0000
1,materials,OUT_OF_RANGE_PRICE,15,8,0.5333
2,po_lines,ARITHMETIC_INCONSISTENCY,807,791,0.9802
3,po_lines,BROKEN_FK,359,359,1.0000
4,po_lines,INVALID_UOM,717,715,0.9972
5,po_lines,MISSING_CRITICAL,1076,1076,1.0000
6,po_lines,NEGATIVE_QUANTITY,359,359,1.0000
7,po_lines,TYPE_MISMATCH,448,448,1.0000
8,po_lines,ZERO_PRICE,538,538,1.0000
9,purchase_orders,DATE_ORDER_VIOLATION,141,141,1.0000


In [9]:
total_recall = recall["detected"].sum() / recall["injected"].sum()
full = (recall["recall"] == 1.0).sum()
print(f"Overall recall: {total_recall:.2%} "
      f"({recall['detected'].sum():,} of {recall['injected'].sum():,} injected defects)")
print(f"{full} of {len(recall)} defect families recovered completely")

Overall recall: 99.46% (4,640 of 4,665 injected defects)
11 of 14 defect families recovered completely


In [10]:
precision = evaluate_rule_precision(ledger, findings)
deterministic = precision[~precision["is_statistical"]]
statistical = precision[precision["is_statistical"]]

print(f"Deterministic rules: {len(deterministic)} rules, "
      f"{deterministic['flagged'].sum():,} cells flagged, "
      f"precision {deterministic['true_positives'].sum() / deterministic['flagged'].sum():.2%}\n")
deterministic[["table", "rule_id", "flagged", "true_positives", "precision"]]

Deterministic rules: 16 rules, 6,307 cells flagged, precision 100.00%



,table,rule_id,flagged,true_positives,precision
0,materials,expect_column_values_to_match_regex.description,50,50,1.0
1,po_lines,consistency.material_id_exists_in_material_master,359,359,1.0
2,po_lines,consistency.net_value_matches_quantity_times_p...,1669,1669,1.0
3,po_lines,consistency.receipt_not_greater_than_order,359,359,1.0
4,po_lines,consistency.uom_matches_material_master,715,715,1.0
5,po_lines,expect_column_values_to_be_between.quantity,359,359,1.0
6,po_lines,expect_column_values_to_be_between.unit_price,538,538,1.0
7,po_lines,expect_column_values_to_be_of_type.quantity,448,448,1.0
8,po_lines,expect_column_values_to_not_be_null.unit_price,1076,1076,1.0
9,purchase_orders,consistency.delivery_date_after_order_date,181,181,1.0


Every deterministic rule is exactly right: if the profile says `unit_price` must be
present and positive, then every cell it flags really was corrupted. That is expected -
a rule is a definition, not a prediction - but asserting it catches the rules that are
silently matching the wrong rows.

The statistical rules are a different matter, and are reported separately for that
reason.

In [11]:
statistical[["table", "rule_id", "flagged", "true_positives", "precision"]]

,table,rule_id,flagged,true_positives,precision
16,po_lines,expect_column_values_to_be_plausible.net_value,2,0,0.0000
17,purchase_orders,expect_column_values_to_be_plausible.total_net...,36,2,0.0556
18,po_lines,expect_column_values_to_be_plausible.quantity,905,96,0.1061
19,po_lines,expect_column_values_to_be_plausible.unit_price,13,2,0.1538
20,materials,expect_column_values_to_be_plausible.standard_...,9,8,0.8889


The outlier rule recovers roughly half of the fat-fingered prices at a cut of 4.0
robust standard deviations, with the rest of its flags being legitimate extremes. That
is the honest trade-off and it is why outliers carry the lowest dimension weight and
the lowest severity confidence: an outlier is a *suspicion*, not a violation.

One further detail matters here. Prices and quantities are log-normal, so a robust
z-score on the raw scale flags the entire right tail of a healthy column. Applying it
on the log scale instead cut the false positives on `unit_price` from 10,089 to 13
without losing a single true defect.

## The deliverable: a prioritized register

In [12]:
from dq_anomaly.pipeline import AuditConfig, audit_batch

result = audit_batch(
    dirty["po_lines"],
    profile=load_profile("profile_erp_po_lines.yaml"),
    reference_tables=dirty,
    table_name="erp_po_lines",
    batch_id="notebook-demo",
    config=AuditConfig(use_model=False),
)
issues = result.report.issues_frame()
print(f"Quality score {result.quality.global_score:.2f} (grade {result.quality.grade}), "
      f"{len(issues)} issues")
issues[["priority", "severity", "dimension", "rule_id", "n_records"]].head(10)

Quality score 99.77 (grade A), 9 issues


,priority,severity,dimension,rule_id,n_records
0,1,high,completeness,expect_column_values_to_not_be_null.unit_price,1076
1,2,high,validity,expect_column_values_to_be_between.unit_price,538
2,3,high,validity,expect_column_values_to_be_between.quantity,359
3,4,high,validity,expect_column_values_to_be_of_type.quantity,448
4,5,high,consistency,consistency.material_id_exists_in_material_master,359
5,6,high,consistency,consistency.net_value_matches_quantity_times_p...,1669
6,7,medium,consistency,consistency.uom_matches_material_master,715
7,8,medium,consistency,consistency.receipt_not_greater_than_order,359
8,9,low,plausibility,expect_column_values_to_be_plausible.quantity,905


In [13]:
for issue in result.report.issues[:4]:
    print(f"[{issue.priority}] {issue.severity.value.upper()} - {issue.rule_id}")
    print(f"    {issue.reason}")
    print(f"    -> {issue.recommended_action}\n")

[1] HIGH - expect_column_values_to_not_be_null.unit_price
    'unit_price' is 1.20% null (tolerance 0.00%). 1,076 of 89,649 rows checked failed (1.20%).
    -> Trace the null rows back to the source system and make the field mandatory at entry; quarantine affected records until the value is supplied.

[2] HIGH - expect_column_values_to_be_between.unit_price
    'unit_price' must fall within [0.01, None]. 538 of 88,573 rows checked failed (0.61%).
    -> Reject the out-of-range values at ingestion and add a bounds check to the source form; review whether a sign or unit error is involved.

[3] HIGH - expect_column_values_to_be_between.quantity
    'quantity' must fall within [0.01, None]. 359 of 89,201 rows checked failed (0.40%).
    -> Reject the out-of-range values at ingestion and add a bounds check to the source form; review whether a sign or unit error is involved.

[4] HIGH - expect_column_values_to_be_of_type.quantity
    'quantity' must parse as float. 448 of 89,649 rows checked

Each row carries the reason it was raised, how many records it affects, and what to
do about it. That is the artifact a data quality function hands to whoever owns the
source system - not a score, and not a chart.